# 01 - Getting started with Delphos

This notebook shows how to use Delphos to generate and estimate discrete choice model specifications.

You will learn how to:

- list available datasets
- load a pretrained Delphos
- generate and estimate candidate specifications
- inspect model terms, parameters, and generated Apollo code
- save the resulting proposal table for further analysis

### 1. Import Delphos


In [2]:
import delphos as dp

print("Delphos is ready")


Delphos Backend: R version 4.5.1 (2025-06-13) | Apollo v0.3.7
Delphos is ready


### 2. Explore datasets

`Delphos` includes a collection of discrete choice datasets used during training and evaluation.  You can explore them using `list_datasets()`.



In [7]:
datasets = dp.list_datasets()
for item in datasets:
    print(f"{item.id:>2} | {item.name:<24} | {item.path}")

 1 | ApolloModeChoice         | /Users/gnova/Developer/Main-Delphos/Delphos/src/delphos/data/bundled/datasets/dataset_1
 2 | SwissmetroRouteChoice    | /Users/gnova/Developer/Main-Delphos/Delphos/src/delphos/data/bundled/datasets/dataset_2
 3 | Decisions                | /Users/gnova/Developer/Main-Delphos/Delphos/src/delphos/data/bundled/datasets/dataset_3
 4 | Swissmetro               | /Users/gnova/Developer/Main-Delphos/Delphos/src/delphos/data/bundled/datasets/dataset_4
 5 | NLModeChoice             | /Users/gnova/Developer/Main-Delphos/Delphos/src/delphos/data/bundled/datasets/dataset_5
 6 | NorwayVTT                | /Users/gnova/Developer/Main-Delphos/Delphos/src/delphos/data/bundled/datasets/dataset_6
 7 | Arentze2013              | /Users/gnova/Developer/Main-Delphos/Delphos/src/delphos/data/bundled/datasets/dataset_7
 8 | SpainParkingchoice       | /Users/gnova/Developer/Main-Delphos/Delphos/src/delphos/data/bundled/datasets/dataset_8
 9 | LondonModeChoice         | /Users/g

In this example, we use Swissmetro as an unseen dataset. 

You can see information such as the available alternatives, attributes, socio-demographic variables, number of observations, and panel structure. 

In [29]:
dataset = dp.load_dataset("Swissmetro")
print(dataset)
print(f"N_obs:          {dataset.n_obs}")
print(f"Panel:          {dataset.is_panel}")
print(f"Alternatives:   {dataset.alternatives}")
print(f"Attributes:     {dataset.attributes}")
print(f"Covariates:     {dataset.covariates}")



Task(name='Swissmetro', alternatives=3, attributes=5, covariates=11)
N_obs:          5409
Panel:          True
Alternatives:   (Alternative(id=1, name='TRAIN', choice=1, availability='train_av'), Alternative(id=2, name='SM', choice=2, availability='sm_av'), Alternative(id=3, name='CAR', choice=3, availability='car_av'))
Attributes:     (Attribute(id=1, name='ASC', alternative={}), Attribute(id=2, name='time', alternative={1: 'train_tt_scaled', 2: 'sm_tt_scaled', 3: 'car_tt_scaled'}), Attribute(id=3, name='cost', alternative={1: 'train_cost_scaled', 2: 'sm_cost_scaled', 3: 'car_co_scaled'}), Attribute(id=4, name='headway', alternative={1: 'train_he_scaled', 2: 'sm_he_scaled'}), Attribute(id=6, name='seat', alternative={2: 'sm_seats_scaled'}))
Covariates:     (Covariate(id=4, name='purpose', levels=(1, 2)), Covariate(id=6, name='first', levels=(0, 1)), Covariate(id=None, name='ticket', levels=(1, 2, 3, 4, 5, 6, 7, 8, 10)), Covariate(id=None, name='who', levels=(0, 1, 2, 3)), Covariate(id

### 3. Load the pretrained Delphos agent.

In [30]:
agent = dp.load_agent()

print(agent.agent.summary())

{'agent': 'DelphosAgent', 'mode': 'inference', 'encoder_kind': 'deepset', 'state_dim': 64, 'num_actions': 297, 'z_cfg': {'K': 7, 'T': 3, 'G': 2, 'C': 7, 'd_att': 16, 'd_tr': 8, 'd_taste': 8, 'd_cov': 16, 'd_term': 64, 'd_state': 128, 'context_dim': 0, 'head_flag': False, 'pooling': 'mean', 'attention_heads': 4, 'attention_layers': 1, 'attention_dropout': 0.0}, 'device': 'cpu'}


### 4. Propose and estimate models

Delphos can operate in two modes:

- `estimate=False`: the agent searches the model specification space and generates Apollo-ready candidate specifications without estimating them in R. 
- `estimate=True`: Delphos generates the candidate specifications and estimates them using Apollo in R, returning the corresponding estimation results.

In this example, we use `estimate=True` to generate and estimate the models proposed by the trained agent.

In [31]:
models = agent.propose(
    dataset,
    n_models=5,
    estimate=True
)

In [35]:
models.to_dataframe()

,task_id,task_name,specification_key,episode_length,search_strategy,attempt_found,estimated,reward,n_terms,action_indices,...,rho2_0,adjRho2_0,rho2_C,adjRho2_C,AIC,BIC,eigValue,timeTaken,nFreeParams,skipped
0,4,Swissmetro,1110_2120_3212_4111_5000_6126_7000,10,topk,0,True,0.253664,5,"[125, 74, 17, 149, 27, 131, 75, 215, 106, 17]",...,0.252815,0.250472,0.118513,0.116175,8317.182350,8402.928004,-2.466354,1.822415,13,0
1,4,Swissmetro,1110_2120_3212_4214_5000_6126_7000,10,topk,1,True,0.241411,5,"[21, 74, 125, 27, 105, 215, 17, 73, 75, 125]",...,0.240090,0.237746,0.103500,0.101161,8458.394790,8544.140444,-2.624755,1.857519,13,0
2,4,Swissmetro,1110_2120_3212_4110_5000_6110_7000,10,topk,2,True,0.239392,5,"[215, 27, 17, 106, 21, 73, 17, 105, 75, 201]",...,0.238001,0.236018,0.101036,0.099123,8477.570734,8550.124749,-2.681935,1.454101,11,0
3,4,Swissmetro,1110_2120_3210_4222_5000_6110_7000,10,topk,3,True,0.259389,5,"[17, 27, 125, 21, 74, 17, 27, 131, 17, 73]",...,0.258789,0.256086,0.125561,0.122797,8254.890684,8353.827976,-1.133008,1.712976,15,0
4,4,Swissmetro,1110_2120_3211_4111_5000_6110_7000,10,topk,4,True,0.249923,5,"[21, 27, 74, 17, 21, 125, 215, 106, 201, 17]",...,0.248921,0.247118,0.113919,0.112218,8354.399319,8420.357514,-11.241843,1.345932,10,0


### 5. Inspect candidates

Delphos returns a pool of candidate specifications. You can select any candidate for further inspection.

1. Select a candidate from the proposal pool:

In [37]:
candidate = models.proposals[0] 

2. Inspect the corresponding Apollo specification:

    ```bash
    apollo_spec = candidate.apollo_specification
    ```

In [39]:
apollo_spec = candidate.apollo_specification

In [61]:
print("Number of parameters:", apollo_spec.n_parameters)

apollo_spec.parameter_names

Number of parameters: 14


['ASC_TRAIN',
 'ASC_SM',
 'ASC_CAR',
 'b_TRAIN_time',
 'b_SM_time',
 'b_CAR_time',
 'b_cost_generic_log_income_1',
 'b_cost_generic_log_income_2',
 'b_cost_generic_log_income_3',
 'b_cost_generic_log_income_4',
 'b_headway_generic_male_0',
 'b_headway_generic_male_1',
 'b_SM_seat_first_0',
 'b_SM_seat_first_1']

In [58]:
print("Utility code preview:\n")
print(apollo_spec.utility_code)

Utility code preview:

V <- list()

V[["TRAIN"]] <-
      ASC_TRAIN +
      b_TRAIN_time * train_tt_scaled +
      b_cost_generic_log_income_1 * (income == 1) * log(1+train_cost_scaled) +
      b_cost_generic_log_income_2 * (income == 2) * log(1+train_cost_scaled) +
      b_cost_generic_log_income_3 * (income == 3) * log(1+train_cost_scaled) +
      b_cost_generic_log_income_4 * (income == 4) * log(1+train_cost_scaled) +
      b_headway_generic_male_0 * (male == 0) * train_he_scaled +
      b_headway_generic_male_1 * (male == 1) * train_he_scaled

V[["SM"]] <-
      ASC_SM +
      b_SM_time * sm_tt_scaled +
      b_cost_generic_log_income_1 * (income == 1) * log(1+sm_cost_scaled) +
      b_cost_generic_log_income_2 * (income == 2) * log(1+sm_cost_scaled) +
      b_cost_generic_log_income_3 * (income == 3) * log(1+sm_cost_scaled) +
      b_cost_generic_log_income_4 * (income == 4) * log(1+sm_cost_scaled) +
      b_headway_generic_male_0 * (male == 0) * sm_he_scaled +
      b_headway_gen

### 6. Save candidates


In [62]:
output_path = "getting_started_proposals.csv"
models.to_dataframe().to_csv(output_path, index=False)
print(f"Saved {len(models)} proposals to {output_path}")


Saved 5 proposals to getting_started_proposals.csv
